# 10. Attack checks and statistics

Produces, from the saved results:

- **Table S-XXI** (`reports/tables/budget_check.csv`): five-times stronger APGD and Square against the grid.
- **Tables S-XXII and S-XXIII** (`reports/table8_aats_vs_milp.csv`, rows by `mode`): AATS against the exact MILP.
- **Tables S-XXXVIII and S-XXXIX** (`reports/table6_mixed_model.csv`, `table6_friedman.csv`): Bayesian mixed model and
  Friedman tests.
- **Table S-XL** (`reports/table11_halves.csv`, mean per dataset in `reports/tables/halves_by_dataset.csv`): the two
  halves of the fraud sample.
- **Table S-XLI** (`reports/table14_rerun_by_attack.csv`): first grid against the final grid.
- **Table S-XLII** (`reports/table15_hsj_starts.csv`): five-start HopSkipJump.

Inputs: the attack grid (`runs/<d>/attacks/results*.csv`, `per_example*.npz`, `attack_scores*.npz`, `sample.npz`), the
first grid (`runs/<d>/archive_half/`, `archive_attacks_20261114/`), `certify_*.csv`, `budget_check*.csv`,
`hsj_starts*.csv` and the models' `metrics.json`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))   # the library (not needed after pip install -e .)
from fraudadv import notebook as nb
ROOT = nb.setup()          # works from the folder root, sets the study's environment

The summary functions write into `reports/`; create the folders as `scripts/make_results.sh` does, and load the
attack grid (every `runs/<d>/attacks/results*.csv` in one frame).

In [ ]:
from fraudadv.report import make_report as MR
from fraudadv.report import tables

(ROOT / "reports" / "tables").mkdir(parents=True, exist_ok=True)
A = MR.load_attacks()
print(len(A), "rows of the attack grid")

## Table S-XXI

`tables.budget_check` reads `runs/<d>/attacks/budget_check*.csv` (`attacks/budget_check.py`): worst-case ASR of the
grid against the grid plus APGD with 5x the iterations and Square with 5x the queries, on the same frauds, into
`reports/tables/budget_check.csv`.

In [ ]:
tables.budget_check().head(12)

## Tables S-XXII and S-XXIII

`make_report.certify_table` compares the heuristic tree attack AATS with the exact MILP optimum
(`runs/<d>/attacks/certify_*.csv`, `attacks/certify.py`) per dataset, model and threat model
(`reports/table8_aats_vs_milp.csv`). Table S-XXII is the rows of the grid's threat models (`mode` constrained and
unconstrained); Table S-XXIII is the first, continuous certification (`mode` continuous).

In [ ]:
import pandas as pd

MR.certify_table()
T8 = pd.read_csv(ROOT / "reports/table8_aats_vs_milp.csv")
T8[T8["mode"] != "continuous"].head(12)

In [ ]:
T8[T8["mode"] == "continuous"]

## Tables S-XXXVIII and S-XXXIX

`make_report.stats_tables` fits the Bayesian mixed model of per-example evasion at eps = 0.25 (worst case, detected
frauds, dataset random intercepts) and the Friedman test with Nemenyi critical differences over (dataset, setting)
blocks, and writes the McNemar pairs and Wilson intervals (`reports/table6_*.csv`). The mixed model's start values are
random (statsmodels), which can move its estimates by about 5e-7; the CSV is rounded to 4 decimals, so it normally
matches the reference exactly (the check below says whether it does).

In [ ]:
MR.stats_tables(A)
nb.show_csv("reports/table6_mixed_model.csv")
nb.show_csv("reports/table6_friedman.csv");

## Table S-XL

`make_report.halves_table`: detection and worst-case ASR on the first and second half of each entity-ordered fraud
sample, per model (`reports/table11_halves.csv`); `tables.halves_by_dataset` gives the mean over the models of each
dataset (`reports/tables/halves_by_dataset.csv`, the top rows of the table).

In [ ]:
MR.halves_table()
tables.halves_by_dataset()

## Table S-XLI

`make_report.rerun_table` compares the first grid with the whole-sample rerun outcome by outcome
(`reports/table13_rerun_agreement.csv`); `make_report.rerun_attack_table` does it attack by attack
(`reports/table14_rerun_by_attack.csv`, Table S-XLI).

In [ ]:
MR.rerun_table()
MR.rerun_attack_table()
nb.show_csv("reports/table14_rerun_by_attack.csv", rows=6);

## Table S-XLII

`make_report.hsj_starts_table` reads `runs/<d>/attacks/hsj_starts.csv` (`attacks/hsj_starts.py`): HopSkipJump from one
seeded start and from five, beside the grid's Square attack, into `reports/table15_hsj_starts.csv`.

In [ ]:
MR.hsj_starts_table()
nb.show_csv("reports/table15_hsj_starts.csv", rows=12);

## Rerun the runs behind these results from scratch

The calls `scripts/reproduce.sh` makes for this step, for one dataset: certification of the tree attack by the exact
MILP in both threat models (`attacks/certify.py`), the attack-strength check (`attacks/budget_check.py`) and the
five-start HopSkipJump (`attacks/hsj_starts.py`). The continuous certification of Table S-XXIII was a first run on the
first grid (`--mode continuous`, the module's default); it is listed commented out. The attack grid itself
(`attacks/run_attacks.py`), input of every table here, is not repeated in this cell. They take hours and write into `runs/<dataset>/attacks/`;
they skip work whose outputs exist. Not run here.

In [ ]:
RERUN = False
DATASET = "ulb"
if RERUN:
    import os
    import subprocess

    def py(*args):
        """python3 -m <module> <args> from the folder root, as py() in scripts/reproduce.sh."""
        subprocess.run([sys.executable, "-m", *args], check=True, cwd=ROOT,
                       env=dict(os.environ, PYTHONPATH=str(ROOT / "src")))

    # what runs on this dataset (as in scripts/reproduce.sh)
    TAB = "xgboost,catboost,mlp,ft_transformer,ivis"
    CTX = "" if DATASET in ("ulb", "baf") else "hgnn,hgt"
    FMS = "" if DATASET in ("ulb", "baf") else "chronos2,timesfm"
    ALL = TAB + ("," + CTX if CTX else "")
    CONSISTENT = DATASET in ("ieee_cis", "sparkov", "paysim")

    for mode in ("constrained", "unconstrained"):
        py("fraudadv.attacks.certify", DATASET, "--mode", mode)
    # py("fraudadv.attacks.certify", DATASET, "--mode", "continuous")    # Table S-XXIII: the first, continuous run
    py("fraudadv.attacks.budget_check", DATASET, ALL + ("," + FMS if FMS else ""))
    py("fraudadv.attacks.hsj_starts", DATASET, ALL)

## Check against the reference build

Every file this notebook wrote, compared byte for byte with the same path under `$FRAUDADV_REFERENCE`.

In [ ]:
nb.check([
    "reports/tables/budget_check.csv",
    "reports/table8_aats_vs_milp.csv", "reports/table8_aats_vs_milp.md",
    "reports/table6_worstcase_ci.csv", "reports/table6_friedman.csv", "reports/table6_mcnemar.csv",
    "reports/table6_mixed_model.csv", "reports/table6_mixed_model_nopaysim.csv",
    "reports/table11_halves.csv", "reports/tables/halves_by_dataset.csv",
    "reports/table13_rerun_agreement.csv", "reports/table14_rerun_by_attack.csv",
    "reports/table14_rerun_by_attack_cells.csv",
    "reports/table15_hsj_starts.csv",
])